# L10 · NB 01 — Monday morning: feel the power of pretrained transformers

> ⏱️ **~20 min** · pre-class · runs on CPU (first run downloads models — allow extra time on slow wifi)

> *Marcus's Friday question is on Sarah's mind: build a shopping assistant. Before she designs anything, she opens her laptop and explores what pretrained transformer models can already do — out of the box, in one line each.*

This pre-class notebook is your hands-on introduction to the modern NLP stack. We will use three pretrained pipelines:

1. **Sentiment analysis** — is a review positive or negative?
2. **Named-entity recognition (NER)** — extract people, places, organisations
3. **Zero-shot classification** — sort text into labels you describe, with no training

Each is one line of Python. Each runs a 67-400M parameter transformer underneath. Welcome to the era of pretrained models.

---

### Where you are in L10

Marcus asked for a shopping assistant. Sarah can't build it in one leap — she works through four notebooks, each answering the next question in the chain:

| Step | Notebook | Sarah's question |
|---|---|---|
| 1 | `01_monday_morning` 👈 **you are here** | What can pretrained transformers already do, out of the box? |
| 2 | `02_attention_intuition` | *Why* do they work? What is this "attention" everyone talks about? |
| 3 | `03_using_an_llm` | How do I drive a generative LLM myself — tokens, sampling, chat? |
| 4 | `04_rag_pipeline` | How do I ground it in NorthStar's catalogue? → the shopping assistant |

Right now you're on **step 1 of 4**. Each notebook stands on the one before it — run them in order.


## 1 · Setup


> **Why this cell first — and why it looks like a pile of switches.**
> Before Sarah touches a single model she runs this setup cell. It isn't glamorous, but every line earns its place:
>
> - `KMP_DUPLICATE_LIB_OK='TRUE'` — on macOS, PyTorch and the other scientific libraries each ship their own copy of the OpenMP maths runtime. When two copies load at once the kernel **crashes the instant you `import torch`**. This line tells them to coexist instead of fighting. (It's the same fix the repo's `.env` file applies — we repeat it here so the notebook also works in Colab and on a fresh machine.)
> - `HF_HUB_DISABLE_TELEMETRY` / `TRANSFORMERS_VERBOSITY='error'` / `warnings.filterwarnings('ignore')` — silence the download chatter and version warnings so the output you see is the output that *matters*.
>
> You'll see this same opening cell in every model notebook. Run it once at the top — it's the seatbelt before the drive.


In [1]:
# --- Environment setup: run this once before loading any model ---
import os
# Tell the OpenMP runtime to tolerate duplicate copies (avoids a macOS crash on import torch)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'  # don't phone home to Hugging Face
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'  # only show real errors, hide info/warnings
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # prevent tokenizer deadlock on macOS + VS Code

# Hide library warnings so the output we care about stays readable
import warnings
warnings.filterwarnings('ignore')

# torch = the deep-learning engine; pipeline = the one-line helper that loads + runs a model
import torch
from transformers import pipeline

# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)
print('Torch:', torch.__version__)  # confirm PyTorch imported and show its version

Torch: 2.12.0


## 2 · Sentiment analysis

The classic NLP starter. Is this product review positive or negative?

In [2]:
# pipeline('sentiment-analysis') downloads a pretrained model (first run only)
# and returns a ready-to-call function that labels text POSITIVE or NEGATIVE.
sentiment = pipeline('sentiment-analysis')

# A handful of product reviews to test on, ranging from clearly happy to clearly upset.
reviews = [
    "I love this dress! Perfect for a beach holiday and the fabric is so soft.",
    "Returned. The colour in the photo was MUCH brighter than what arrived.",
    "It's fine. Does the job. Nothing special.",
    "Worst purchase I've made this year. Stitching unravelled within two wears.",
    "Got it as a gift and she loves it — thank you so much!",
]

# Run the model on each review one at a time.
for r in reviews:
    # sentiment(r) returns a list of dicts; [0] pulls out the single prediction for this text.
    # Each dict has 'label' (POSITIVE/NEGATIVE) and 'score' (the model's confidence, 0-1).
    result = sentiment(r)[0]
    # Print the label, confidence, and the first 60 characters of the review for context.
    print(f"  [{result['label']:<8s} {result['score']:.3f}]  {r[:60]}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  [POSITIVE 1.000]  I love this dress! Perfect for a beach holiday and the fabri
  [POSITIVE 0.998]  Returned. The colour in the photo was MUCH brighter than wha
  [NEGATIVE 0.956]  It's fine. Does the job. Nothing special.
  [NEGATIVE 1.000]  Worst purchase I've made this year. Stitching unravelled wit
  [POSITIVE 1.000]  Got it as a gift and she loves it — thank you so much!


**One line of Python** — and notice the result is mostly right, but not perfectly so. Reread review 2: *"Returned. The colour in the photo was MUCH brighter than what arrived."* That's a complaint. The model labelled it POSITIVE with 0.998 confidence. The word *brighter* and the absence of explicitly negative words tricked it.

This is a real failure mode of off-the-shelf models. They're shockingly capable on the easy 90%, and they hallucinate confidence on the 10%. Production use means knowing where the gaps are.

What's running underneath? `pipeline('sentiment-analysis')` defaults to `distilbert-base-uncased-finetuned-sst-2-english` — a 67M-parameter transformer fine-tuned on the Stanford Sentiment Treebank (movie reviews). Loads ~268 MB the first time. Because it was trained on movie reviews, it does fine on product reviews that LOOK like movie reviews but struggles on retail-specific patterns like "I returned this". A small fine-tune on YOUR review corpus would fix this.

That's the modern NLP stack: someone has trained a model for you; you import and use it; and you ALWAYS run a calibration pass on your own data.

## 3 · Named-entity recognition (NER)

Extract people, places, organisations, dates from text. Useful for tagging news articles, customer support tickets, product descriptions.

In [3]:
# Load a NER model. We name the model explicitly ('dslim/bert-base-NER') instead of using
# the default. aggregation_strategy='simple' merges word-pieces into whole entities
# (so "Sarah Chen" comes back as one name, not split fragments).
ner = pipeline('ner', model='dslim/bert-base-NER', aggregation_strategy='simple')

# A sentence containing several real-world entities (people, an organisation, a place).
text = "Sarah Chen from NorthStar Retail launched a new search feature in London on Monday. The team thanked Marcus Patel for sponsoring the project."

# Run the model; it returns a list of dicts, one per detected entity.
results = ner(text)
print(f"Input: {text}\n")
# Print a header row, then a divider line, for a tidy table.
print(f"{'Entity':<25s} {'Type':<10s} {'Score':>8s}")
print('-' * 50)
# For each detected entity, show the text ('word'), its category
# ('entity_group': PER=person, ORG=organisation, LOC=location), and confidence.
for r in results:
    print(f"  {r['word']:<25s} {r['entity_group']:<10s} {r['score']:>8.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Input: Sarah Chen from NorthStar Retail launched a new search feature in London on Monday. The team thanked Marcus Patel for sponsoring the project.

Entity                    Type          Score
--------------------------------------------------
  Sarah Chen                PER           1.000
  NorthStar Retail          ORG           0.999
  London                    LOC           1.000
  Marcus Patel              PER           1.000


The model isn't perfect (it might miss a name, mis-classify a city), but it's getting most of these right with **zero training on your data**. NER is a textbook task; pretrained models nail it.

Production use cases: redacting PII (Personal Identifiable Information) from logs, tagging support tickets for routing, extracting people/companies from news for a feed.

## 4 · Zero-shot classification

The third demo is more impressive than it sounds. **Zero-shot** means you classify text into categories the model has never been explicitly trained on — you just *describe* the labels and the model decides which one fits.

In [4]:
# Load a zero-shot classifier. This model can sort text into ANY labels you supply,
# without being trained on them first.
zsc = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

# The text to classify...
text = "Returned the shirt after one wash — the colour ran and now my whites are pink."
# ...and the candidate categories we invent on the spot (just strings, no training needed).
candidate_labels = ["quality complaint", "praise", "shipping issue", "size issue", "billing question"]

# Run it: the model scores how well each label fits, returning labels sorted best-first.
result = zsc(text, candidate_labels)
print(f"Input: {text}\n")
print(f"{'Label':<25s} {'Score':>8s}")
print('-' * 36)
# result['labels'] and result['scores'] are parallel lists; zip pairs each label with its score.
# Scores sum to ~1.0 across all labels — higher means a better fit.
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:<25s} {score:>8.3f}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Input: Returned the shirt after one wash — the colour ran and now my whites are pink.

Label                        Score
------------------------------------
  quality complaint            0.717
  billing question             0.174
  shipping issue               0.065
  size issue                   0.027
  praise                       0.018


Same model, completely different label set — just by changing the candidate strings. This is shockingly useful in production: you can build a customer-support ticket router in 30 minutes without labelled training data.

Underneath, the model (`bart-large-mnli`) was trained on natural-language-inference data — predicting whether a hypothesis is *entailed*, *contradicted*, or *neutral* given a premise. Zero-shot classification reformulates "is this text about X?" as "does this text entail the statement 'This is about X'?". Clever.

**The pattern emerging:** every one of these used to require custom training. Today: pip install, import, call. The models are pretrained, instruction-tuned, fine-tuned, and waiting on Hugging Face.

## 5 · The pattern: hub + pipeline

Hugging Face hosts **hundreds of thousands of pretrained models** for every NLP task imaginable. The `pipeline()` function from `transformers` is the simplest API:

```python
from transformers import pipeline
model = pipeline("task-name")           # uses default model for the task
model = pipeline("task-name", model=    # specify any model from the Hub
                 "user/model-id")
result = model(input_text)              # run it
```

Available tasks include:

- `sentiment-analysis`, `text-classification`
- `ner`, `token-classification`
- `zero-shot-classification` (label without training data)
- `fill-mask` (BERT-style "Paris is the [MASK] of France")
- `feature-extraction` (raw embeddings)
- `text-generation` (LLMs — we'll meet these in NB 03)
- `image-classification`, `image-segmentation`, `object-detection`
- `automatic-speech-recognition`, `text-to-speech`

You can build a surprising amount of product on these pipelines alone. No model training required.

## 6 · Three thought-questions for class

1. The sentiment pipeline labelled reviews 1, 2, 3, 4, 5 with high confidence. **Predict** what it would return for: *"It's fine, I guess. Wouldn't buy again."* — then run it and see if you were right.
2. The NER model knew "Sarah Chen" was a PER (person), "NorthStar Retail" was an ORG, "London" was a LOC. **How** do you think it learnt those distinctions? What did its training data look like?
3. If you wanted to build a product-review sentiment classifier specific to YOUR domain (e.g., recognising that "the elastic was loose" is negative), would you (a) prompt a general LLM, (b) use this off-the-shelf pipeline, (c) fine-tune a small model on your reviews? What's the trade-off?

In [5]:
# Reuse the sentiment pipeline from section 2 on your own sentence.
# Try editing this string and predict the label before running.
test = "It's fine, I guess. Wouldn't buy again."
# Prints the raw output: a list with one dict holding the predicted label and confidence.
print(sentiment(test))

[{'label': 'POSITIVE', 'score': 0.9971808195114136}]


---

**Ready for class.** You've felt the power of pretrained transformers. Tomorrow we'll open the box and see WHY they work — the attention mechanism that makes all of this possible.